# Exemplo: Active Learning no SVHN (sem oráculo real)

Exemplo de ponta a ponta da biblioteca usando o `SVHNCustomCNN` (`Models/models.py`) e os dados em `Data/Example_SVHN/`.

Cenário simulado: **não existe oráculo real disponível**. Apenas **20%** do conjunto de treino começa rotulado (rótulo verdadeiro, vindo do CSV original). O restante é pool não rotulado — e quando uma amostra dele precisa de rótulo, quem faz esse papel é o próprio pseudo-labeling, não um humano.

Fluxo por ciclo (10 épocas de treino cada):
1. **Query** (`uncertainty_query_strategy`) escolhe candidatos do pool não rotulado (as amostras mais incertas).
2. **Pseudo-labeling** (`pseudo_labeling_strategy`) atua como "oráculo" só para esses candidatos, com um threshold de confiança mais permissivo (`50%`, já que não há outra fonte de rótulo disponível).
3. **Balance** (`class_balance_strategy`) evita que uma classe domine os candidatos aceitos.
4. Os aceitos são gravados **permanentemente** numa base CSV nova (`train_self_labeled.csv`) que só cresce — o `train.csv` original nunca é alterado. Os rejeitados voltam pro pool não rotulado e podem ser escolhidos de novo em ciclos futuros.

A cada ciclo, salvamos em disco as imagens escolhidas pela query, o resumo do balanceamento e as imagens/rótulos aceitos permanentemente.

In [1]:
import sys
sys.path.append("Utils/Example_SVHN")
sys.path.append("Models")
sys.path.append("Query_Strategies")
sys.path.append("Labeling_Strategies")
sys.path.append("Balance_Strategies")
sys.path.append("Training")

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import save_image

from Example_data_utils import SVHNCustomDataset
from models import SVHNCustomCNN
from querys_strategies import uncertainty_query_strategy
from labeling_strategies import pseudo_labeling_strategy
from balance_strategies import class_balance_strategy
from tasks import ClassificationTask
from self_labeling_cycle import run_self_labeling_cycle

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Dados
DATA_DIR = "Data/Example_SVHN/csv_SVHN"
TRAIN_IMG_DIR = "F:/SVHN/train/train/"
TEST_IMG_DIR = "F:/SVHN/test/test/"
IMG_SIZE = (640, 640)
NUM_POSITIONS = 5
NUM_CLASSES = 11  # 0-9 + pad
PAD_TOKEN = 10

# Treino
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-3
SEED = 42

# Active Learning (sem oraculo real)
INITIAL_LABELED_FRACTION = 0.20
NUM_CYCLES = 3           # ajuste livremente para um teste maior/menor
CYCLE_BUDGET = 300       # quantos candidatos a query seleciona por ciclo (nem todos viram rotulados: so os que passarem no threshold de confianca do pseudo-labeling)
CONFIDENCE_THRESHOLD = 0.5  # pseudo-labeling atua como "oraculo": threshold mais permissivo, pois nao ha outra fonte de rotulo disponivel
PSEUDO_MAX_PER_CLASS = None  # None = balanceamento automatico (classe com menos candidatos)

# Saidas
OUTPUT_DIR = "outputs/svhn_example"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "runs")
ARTIFACTS_DIR = os.path.join(OUTPUT_DIR, "cycle_artifacts")
HISTORY_CSV = os.path.join(OUTPUT_DIR, "history.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

Device: cuda


## 1. Dataset e split inicial (20% rotulado / 80% pool não rotulado)

In [3]:
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.439, 0.434, 0.439], std=[0.204, 0.208, 0.208]),
])

train_ds = SVHNCustomDataset(
    os.path.join(DATA_DIR, "train.csv"), TRAIN_IMG_DIR,
    transform=transform, max_len=NUM_POSITIONS, pad_token=PAD_TOKEN,
)
test_ds = SVHNCustomDataset(
    os.path.join(DATA_DIR, "test.csv"), TEST_IMG_DIR,
    transform=transform, max_len=NUM_POSITIONS, pad_token=PAD_TOKEN,
)

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Treino: {len(train_ds)} amostras | Teste: {len(test_ds)} amostras")

Treino: 33402 amostras | Teste: 13068 amostras


In [4]:
rng = np.random.default_rng(SEED)
shuffled_indices = rng.permutation(len(train_ds))

initial_labeled_size = int(len(train_ds) * INITIAL_LABELED_FRACTION)
labeled_indices = shuffled_indices[:initial_labeled_size]
unlabeled_indices = shuffled_indices[initial_labeled_size:]

print(f"Rotulado inicial (20%): {len(labeled_indices)} | Pool nao rotulado: {len(unlabeled_indices)}")

Rotulado inicial (20%): 6680 | Pool nao rotulado: 26722


## 2. Modelo, Task e otimizador

`ClassificationTask` recebe `criterion`/`metric_fn` customizados porque a saída do `SVHNCustomCNN` tem uma dimensão extra de posição (`[Batch, 5, 11]`, uma cabeça por dígito da sequência).

In [5]:
def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")


def model_fn():
    model = SVHNCustomCNN(num_positions=NUM_POSITIONS, num_classes=NUM_CLASSES)
    model.apply(init_weights)
    return model


def optimizer_fn(model):
    return optim.Adam(model.parameters(), lr=LR)


def svhn_criterion(outputs, targets):
    # outputs: [Batch, 5, 11]  targets: [Batch, 5]
    return nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)(outputs.view(-1, NUM_CLASSES), targets.view(-1))


def svhn_sequence_accuracy(outputs, targets):
    # 1.0 se a sequencia inteira de digitos bateu, 0.0 caso contrario (por amostra)
    preds = torch.argmax(outputs, dim=2)
    correct = torch.all(preds == targets, dim=1).float()
    return correct.cpu().tolist()


task = ClassificationTask(criterion=svhn_criterion, metric_fn=svhn_sequence_accuracy)

## 3. Base rotulada persistente + estratégias

Criamos uma base CSV separada (`train_self_labeled.csv`) que começa só com os 20% reais e cresce a cada ciclo com os pseudo-rótulos aceitos — o `train.csv` original nunca é tocado. As três estratégias (`uncertainty_query_strategy`, `pseudo_labeling_strategy`, `class_balance_strategy`) são usadas via wrappers finos que só adicionam o salvamento em disco; a lógica de negócio é 100% da biblioteca.

In [6]:
GROWING_CSV_PATH = os.path.join(OUTPUT_DIR, "train_self_labeled.csv")

original_train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

# Base inicial: só os 20% sorteados como rotulado real, copiados como estão.
# O train.csv original nunca é alterado - essa é uma base NOVA e separada, que vai crescer
# a cada ciclo com os pseudo-rótulos aceitos (permanentemente).
initial_rows = original_train_df.iloc[labeled_indices][["filename", "digits"]].copy()
initial_rows.to_csv(GROWING_CSV_PATH, index=False)

print(f"Base rotulada persistente criada em {GROWING_CSV_PATH} com {len(initial_rows)} amostras (20% reais).")

Base rotulada persistente criada em outputs/svhn_example\train_self_labeled.csv com 6680 amostras (20% reais).


In [7]:
MEAN = torch.tensor([0.439, 0.434, 0.439]).view(3, 1, 1)
STD = torch.tensor([0.204, 0.208, 0.208]).view(3, 1, 1)


def save_selected_images(dataset, indices, out_dir, prefix, pseudo_labels=None):
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    for i, idx in enumerate(indices):
        idx = int(idx)
        image, true_digits, _ = dataset[idx]
        image = (image * STD + MEAN).clamp(0, 1)
        save_image(image, os.path.join(out_dir, f"{prefix}_{idx}.png"))

        row = {"dataset_index": idx, "true_digits": true_digits.tolist()}
        if pseudo_labels is not None:
            pseudo = pseudo_labels[i]
            row["pseudo_digits"] = pseudo.tolist() if torch.is_tensor(pseudo) else pseudo
        rows.append(row)

    pd.DataFrame(rows).to_csv(os.path.join(out_dir, f"{prefix}_manifest.csv"), index=False)


def _label_key(label):
    value = label.tolist() if torch.is_tensor(label) else label
    return tuple(value) if isinstance(value, list) else value


def build_labeled_dataset_fn():
    # Relê o CSV persistente a cada ciclo, pra sempre refletir o que já foi
    # permanentemente auto-rotulado até agora.
    return SVHNCustomDataset(
        GROWING_CSV_PATH, TRAIN_IMG_DIR, transform=transform,
        max_len=NUM_POSITIONS, pad_token=PAD_TOKEN,
    )


cycle_state = {"cycle": -1}


def logging_query_strategy(**kwargs):
    cycle_state["cycle"] += 1
    cycle = cycle_state["cycle"]
    selected = uncertainty_query_strategy(**kwargs)

    out_dir = os.path.join(ARTIFACTS_DIR, f"cycle_{cycle}", "query_selected")
    save_selected_images(train_ds, selected, out_dir, "query")
    print(f"[Ciclo {cycle}] Query (uncertainty) selecionou {len(selected)} candidatos -> {out_dir}")
    return selected


def logging_balance_fn(candidates, max_per_class=None):
    cycle = cycle_state["cycle"]
    balanced = class_balance_strategy(candidates, max_per_class=max_per_class)

    out_dir = os.path.join(ARTIFACTS_DIR, f"cycle_{cycle}", "balance")
    os.makedirs(out_dir, exist_ok=True)

    before_counts = pd.Series([_label_key(c[2]) for c in candidates]).value_counts()
    after_counts = pd.Series([_label_key(c[2]) for c in balanced]).value_counts()
    summary = pd.DataFrame({
        "candidatos_antes_do_balanceamento": before_counts,
        "candidatos_depois_do_balanceamento": after_counts,
    }).fillna(0).astype(int)
    summary.to_csv(os.path.join(out_dir, "balance_summary.csv"))

    print(f"[Ciclo {cycle}] Balance: {len(candidates)} candidatos confiantes (>= {CONFIDENCE_THRESHOLD * 100:.0f}%) -> "
          f"{len(balanced)} balanceados ({summary.shape[0]} classes distintas) -> {out_dir}")
    return balanced


def on_accepted(cycle, accepted_indices, accepted_labels):
    # 1) Salva as imagens + manifest, pra inspeção visual (isso aqui nunca volta pro treino)
    out_dir = os.path.join(ARTIFACTS_DIR, f"cycle_{cycle}", "pseudo_labeled_accepted")
    save_selected_images(train_ds, accepted_indices, out_dir, "accepted", pseudo_labels=accepted_labels)

    # 2) Persiste PERMANENTEMENTE na base rotulada (nunca mexe no train.csv original)
    new_rows = []
    for idx, label in zip(accepted_indices, accepted_labels):
        idx = int(idx)
        digits = label.tolist() if torch.is_tensor(label) else list(label)
        digits = [int(d) for d in digits if d != PAD_TOKEN]
        if not digits:
            continue  # predição degenerada (só pad_token), descarta
        filename = original_train_df.iloc[idx]["filename"]
        new_rows.append({"filename": filename, "digits": str(digits)})

    if new_rows:
        pd.DataFrame(new_rows).to_csv(GROWING_CSV_PATH, mode="a", header=False, index=False)

    print(f"[Ciclo {cycle}] {len(new_rows)} amostras auto-rotuladas persistidas permanentemente "
          f"em {GROWING_CSV_PATH} -> {out_dir}")

## 4. Ciclo de Active Learning

In [8]:
final_unlabeled_indices = run_self_labeling_cycle(
    model_fn=model_fn,
    unlabeled_dataset=train_ds,
    build_labeled_dataset_fn=build_labeled_dataset_fn,
    test_loader=test_loader,
    task=task,
    query_strategy=logging_query_strategy,
    pseudo_labeling_fn=pseudo_labeling_strategy,
    balance_fn=logging_balance_fn,
    unlabeled_indices=unlabeled_indices,
    on_accepted=on_accepted,
    optimizer_fn=optimizer_fn,
    device=DEVICE,
    num_cycles=NUM_CYCLES,
    cycle_budget=CYCLE_BUDGET,
    epochs=EPOCHS,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    max_per_class=PSEUDO_MAX_PER_CLASS,
    batch_size=BATCH_SIZE,
    num_workers=4,
    checkpoint_dir=CHECKPOINT_DIR,
    log_dir=LOG_DIR,
    history_csv=HISTORY_CSV,
)


--- Ciclo 0 | Base rotulada atual: 6680 amostras | Pool não rotulado: 26722 ---


KeyboardInterrupt: 

## 5. Resultado

In [ ]:
final_labeled_df = pd.read_csv(GROWING_CSV_PATH)
print(f"Base rotulada final (20% reais + auto-rotulados persistidos): {len(final_labeled_df)} amostras")
print(f"Pool nao rotulado restante: {len(final_unlabeled_indices)}")

history = pd.read_csv(HISTORY_CSV)
history